# Projet de Détection de la Maladie d'Alzheimer par Signaux EEG
Ce notebook implémente un pipeline complet incluant le téléchargement du dataset OpenNeuro ds004504, l'extraction de caractéristiques non-linéaires et la classification par Machine Learning.

In [12]:
import mne
import mne_bids

# Define the root directory of the BIDS dataset
bids_root = 'ds004504-download'

# Updated parameters for ds004504
subject_id = '001'
task_id = 'eyesclosed'

# Construct the BIDS path (No session for this dataset)
bids_path = mne_bids.BIDSPath(
    subject=subject_id,
    task=task_id,
    datatype='eeg',
    root=bids_root
)

print(f"Reading BIDS data: {bids_path.fpath}")

try:
    raw = mne_bids.read_raw_bids(bids_path=bids_path, verbose=False)
    raw.load_data()

    if raw.info['sfreq'] > 250:
        raw.resample(sfreq=250)

    print("\nRaw data loaded successfully:")
    print(raw)
except Exception as e:
    print(f"Error: {e}")

Reading BIDS data: ds004504-download/sub-001/eeg/sub-001_task-eyesclosed_eeg.set
Reading 0 ... 299899  =      0.000 ...   599.798 secs...


/tmp/ipykernel_485/1384290306.py:22: RuntimeWarning: Did not find any events.tsv associated with sub-001_task-eyesclosed.

The search_str was "ds004504-download/sub-001/**/eeg/sub-001*events.tsv"
  raw = mne_bids.read_raw_bids(bids_path=bids_path, verbose=False)
/tmp/ipykernel_485/1384290306.py:22: RuntimeWarning: Unable to map the following column(s) to to MNE:
Gender: F
Age: 57
Group: A
MMSE: 16
  raw = mne_bids.read_raw_bids(bids_path=bids_path, verbose=False)



Raw data loaded successfully:
<RawEEGLAB | sub-001_task-eyesclosed_eeg.set, 19 x 149950 (599.8 s), ~21.8 MiB, data loaded>


## 1. Exemple d'extraction de Caractéristiques sur 1 Sujet
Traitement des données du dossier `derivatives\sub-001` :
- Chargement des fichiers EEGLAB (.set).
- Segmentation en epochs de 4 secondes.
- Calcul des caractéristiques : Higuchi Fractal Dimension, SVD Entropy, DFA, Zero Crossing Rate et paramètres de Hjorth.

In [13]:
if 'raw' in locals() and raw is not None:
    # Define epoch parameters
    tmin = 0      # start of each epoch in seconds
    tmax = 4      # end of each epoch in seconds (4 seconds duration)
    overlap = 0   # No overlap between epochs

    # Generate events for fixed-length epoching
    # The event_id can be arbitrary for fixed-length epochs
    events = mne.make_fixed_length_events(raw, id=1, start=0, duration=tmax, overlap=overlap)

    # Create epochs
    epochs = mne.Epochs(raw, events, event_id=1, tmin=tmin, tmax=tmax,
                        preload=True, baseline=None, verbose=False)

    print(f"Number of epochs created: {len(epochs)}")
    print(f"Epochs loaded successfully: {epochs}")

    # Optionally, reject bad epochs based on amplitude thresholds (if not done in preprocessing)
    # epochs.drop_bad(reject=dict(eeg=200e-6)) # e.g., 200 uV threshold
    # print(f"Number of epochs after rejection: {len(epochs)}")
else:
    print("Raw data not loaded. Please ensure the previous cell ran successfully.")

Number of epochs created: 149
Epochs loaded successfully: <Epochs | 149 events (all good), 0 – 4 s (baseline off), ~21.6 MiB, data loaded,
 '1': 149>


In [14]:
import antropy as ant
import numpy as np
import pandas as pd

def extract_custom_features(epochs):
    """
    Extracts: Higuchi FD, SVD Entropy, DFA, ZCR, and Hjorth parameters.
    """
    ch_names = epochs.ch_names
    sfreq = epochs.info['sfreq']
    all_epoch_features = []

    print(f"Starting extraction for {len(epochs)} epochs...")

    # Iterate through epochs
    for epoch_idx, data in enumerate(epochs.get_data(copy=False)):
        epoch_feat = {'epoch_idx': epoch_idx}

        # Iterate through channels
        for ch_idx, ch_name in enumerate(ch_names):
            x = data[ch_idx]

            # 1. Higuchi Fractal Dimension
            epoch_feat[f'{ch_name}_hfd'] = ant.higuchi_fd(x)

            # 2. SVD Entropy
            epoch_feat[f'{ch_name}_svd_ent'] = ant.svd_entropy(x, normalize=True)

            # 3. DFA (Detrended Fluctuation Analysis)
            epoch_feat[f'{ch_name}_dfa'] = ant.detrended_fluctuation(x)

            # 4. ZCR (Zero Crossing Rate)
            epoch_feat[f'{ch_name}_zcr'] = ant.num_zerocross(x) / len(x)

            # 5. Hjorth Parameters (Mobility and Complexity)
            # Returns (mobility, complexity)
            hj_mob, hj_comp = ant.hjorth_params(x)
            epoch_feat[f'{ch_name}_hjorth_mobility'] = hj_mob
            epoch_feat[f'{ch_name}_hjorth_complexity'] = hj_comp

        all_epoch_features.append(epoch_feat)
        if (epoch_idx + 1) % 10 == 0:
            print(f"Processed {epoch_idx + 1} epochs...")

    return pd.DataFrame(all_epoch_features)

if 'epochs' in locals():
    final_features_df = extract_custom_features(epochs)
    print("\nFeature extraction complete!")
    display(final_features_df.head())
else:
    print("Please run the segmentation cell first.")

Starting extraction for 149 epochs...
Processed 10 epochs...
Processed 20 epochs...
Processed 30 epochs...
Processed 40 epochs...
Processed 50 epochs...
Processed 60 epochs...
Processed 70 epochs...
Processed 80 epochs...
Processed 90 epochs...
Processed 100 epochs...
Processed 110 epochs...
Processed 120 epochs...
Processed 130 epochs...
Processed 140 epochs...

Feature extraction complete!


,epoch_idx,Fp1_hfd,Fp1_svd_ent,Fp1_dfa,Fp1_zcr,Fp1_hjorth_mobility,Fp1_hjorth_complexity,Fp2_hfd,Fp2_svd_ent,Fp2_dfa,...,Cz_dfa,Cz_zcr,Cz_hjorth_mobility,Cz_hjorth_complexity,Pz_hfd,Pz_svd_ent,Pz_dfa,Pz_zcr,Pz_hjorth_mobility,Pz_hjorth_complexity
0,0,1.665366,0.241657,1.520719,0.000999,0.102471,14.323269,1.657852,0.236015,1.536929,...,1.385748,0.076923,0.090544,15.297774,1.689635,0.272152,1.387847,0.005994,0.095091,14.617525
1,1,1.609528,0.490375,1.412916,0.032967,0.237415,6.176921,1.585769,0.474669,1.415968,...,1.353619,0.000000,0.217368,6.316887,1.666117,0.397143,1.355307,0.057942,0.203469,6.709597
2,2,1.697816,0.322442,1.342456,0.032967,0.204091,7.148493,1.679728,0.276369,1.353100,...,1.378539,0.071928,0.119101,11.491204,1.682217,0.211649,1.366553,0.014985,0.124768,10.936381
3,3,1.726302,0.546019,1.350396,0.088911,0.259304,5.633922,1.712240,0.387356,1.336775,...,1.365618,0.000000,0.232010,5.949030,1.724447,0.489844,1.384921,0.070929,0.213098,6.466001
4,4,1.752699,0.344811,1.289838,0.000000,0.344584,4.219809,1.740166,0.283646,1.303347,...,1.312373,0.000000,0.299241,4.557697,1.714074,0.296887,1.373218,0.000000,0.262711,5.265499


## 2. Extraction de Caractéristiques sur 88 Sujets
Traitement des données du dossier `derivatives`. Pour chaque sujet :
- Chargement des fichiers EEGLAB (.set).
- Segmentation en epochs de 4 secondes.
- Calcul des caractéristiques : Higuchi Fractal Dimension, SVD Entropy, DFA, Zero Crossing Rate et paramètres de Hjorth.

In [15]:
import os
import mne
import pandas as pd
import antropy as ant
import numpy as np

def extract_features_from_data(epochs, subject_id):
    ch_names = epochs.ch_names
    all_epoch_features = []
    data_array = epochs.get_data(copy=False)

    for epoch_idx, data in enumerate(data_array):
        epoch_feat = {'subject_id': subject_id, 'epoch_idx': epoch_idx}
        for ch_idx, ch_name in enumerate(ch_names):
            x = data[ch_idx]
            epoch_feat[f'{ch_name}_hfd'] = ant.higuchi_fd(x)
            epoch_feat[f'{ch_name}_svd_ent'] = ant.svd_entropy(x, normalize=True)
            epoch_feat[f'{ch_name}_dfa'] = ant.detrended_fluctuation(x)
            epoch_feat[f'{ch_name}_zcr'] = ant.num_zerocross(x) / len(x)
            hj_mob, hj_comp = ant.hjorth_params(x)
            epoch_feat[f'{ch_name}_hjorth_mobility'] = hj_mob
            epoch_feat[f'{ch_name}_hjorth_complexity'] = hj_comp
        all_epoch_features.append(epoch_feat)
    return all_epoch_features

# Root path for preprocessed data in derivatives
derivatives_path = 'ds004504-download/derivatives'
subjects = sorted([d for d in os.listdir(derivatives_path) if d.startswith('sub-')])

all_results = []
tmin, tmax = 0, 4

print(f"Found {len(subjects)} subjects. Starting processing...")

for sub in subjects:
    try:
        # Path to the preprocessed .set file
        eeg_dir = os.path.join(derivatives_path, sub, 'eeg')
        set_files = [f for f in os.listdir(eeg_dir) if f.endswith('_eeg.set')]

        if not set_files:
            continue

        file_path = os.path.join(eeg_dir, set_files[0])
        raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)

        # Resample to 250Hz if necessary
        if raw.info['sfreq'] > 250:
            raw.resample(250, verbose=False)

        # Segmentation
        events = mne.make_fixed_length_events(raw, id=1, start=0, duration=tmax, overlap=0)
        epochs = mne.Epochs(raw, events, event_id=1, tmin=tmin, tmax=tmax,
                            baseline=None, preload=True, verbose=False)

        # Extraction
        sub_features = extract_features_from_data(epochs, sub)
        all_results.extend(sub_features)
        print(f"Processed {sub}: {len(epochs)} epochs")

    except Exception as e:
        print(f"Error processing {sub}: {e}")

# Final DataFrame
full_dataset_features = pd.DataFrame(all_results)
display(full_dataset_features.head())
full_dataset_features.to_csv('alzheimer_eeg_features.csv', index=False)
print(f"Processing finished. Total samples: {len(full_dataset_features)}")

Found 88 subjects. Starting processing...
Processed sub-001: 149 epochs
Processed sub-002: 198 epochs


/tmp/ipykernel_485/1939874484.py:45: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)


Processed sub-003: 76 epochs


/tmp/ipykernel_485/1939874484.py:45: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)


Processed sub-004: 176 epochs
Processed sub-005: 201 epochs


...

Processed sub-087: 150 epochs


/tmp/ipykernel_485/1939874484.py:45: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)


Processed sub-088: 196 epochs


,subject_id,epoch_idx,Fp1_hfd,Fp1_svd_ent,Fp1_dfa,Fp1_zcr,Fp1_hjorth_mobility,Fp1_hjorth_complexity,Fp2_hfd,Fp2_svd_ent,...,Cz_dfa,Cz_zcr,Cz_hjorth_mobility,Cz_hjorth_complexity,Pz_hfd,Pz_svd_ent,Pz_dfa,Pz_zcr,Pz_hjorth_mobility,Pz_hjorth_complexity
0,sub-001,0,1.502729,0.330977,1.426604,0.039960,0.117689,6.787678,1.453052,0.307677,...,1.576703,0.017982,0.072464,8.616991,1.275094,0.228637,1.587254,0.017982,0.070682,8.636380
1,sub-001,1,1.534378,0.403814,1.369896,0.039960,0.155547,5.221920,1.478630,0.371968,...,1.558746,0.023976,0.091063,6.765377,1.260560,0.271398,1.553991,0.025974,0.088977,6.631844
2,sub-001,2,1.514846,0.398246,1.396135,0.050949,0.153811,5.155007,1.450618,0.356844,...,1.575630,0.030969,0.080866,7.299270,1.285128,0.250967,1.570211,0.034965,0.079978,7.189297
3,sub-001,3,1.525828,0.396685,1.384782,0.046953,0.151949,5.316032,1.448979,0.340774,...,1.588183,0.016983,0.082215,7.680672,1.292763,0.253214,1.615362,0.020979,0.079420,8.071952
4,sub-001,4,1.475149,0.435728,1.370871,0.055944,0.176147,4.383635,1.426010,0.401530,...,1.519424,0.035964,0.115299,5.491412,1.302568,0.306585,1.594680,0.025974,0.105163,5.763428


Processing finished. Total samples: 17419


## 4. Analyse de la Matrice et Sauvegarde
Vérification des dimensions de la matrice finale et export au format CSV.

In [16]:
if 'full_dataset_features' in locals():
    rows, cols = full_dataset_features.shape
    print(f"Dimension de la matrice de caractéristiques : {rows} lignes (epochs) x {cols} colonnes (features)")
    print(f"\nNombre de sujets traités : {full_dataset_features['subject_id'].nunique()}")
    print(f"Nombre de colonnes par électrode : {(cols - 2) // 19} (HFD, SVD_Ent, DFA, ZCR, Mobility, Complexity)")
    display(full_dataset_features.info())
else:
    print("La matrice n'a pas encore été générée. Veuillez terminer l'exécution de la cellule f9767ebf.")

Dimension de la matrice de caractéristiques : 17419 lignes (epochs) x 116 colonnes (features)

Nombre de sujets traités : 88
Nombre de colonnes par électrode : 6 (HFD, SVD_Ent, DFA, ZCR, Mobility, Complexity)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17419 entries, 0 to 17418
Columns: 116 entries, subject_id to Pz_hjorth_complexity
dtypes: float64(114), int64(1), object(1)
memory usage: 15.4+ MB


None

In [17]:
import os

# Nom du fichier
filename = 'alzheimer_eeg_features.csv'

# Sauvegarde forcée (au cas où)
if 'full_dataset_features' in locals():
    full_dataset_features.to_csv(filename, index=False)
    path = os.path.abspath(filename)
    print(f"Le fichier est sauvegardé avec succès.")
    print(f"Emplacement complet : {path}")
    print(f"Taille du fichier : {os.path.getsize(filename) / 1024 / 1024:.2f} MB")
else:
    print("Erreur : La variable 'full_dataset_features' n'existe pas.")

Le fichier est sauvegardé avec succès.
Emplacement complet : /content/alzheimer_eeg_features.csv
Taille du fichier : 36.58 MB


## 5. Classification Binaire (Alzheimer vs Contrôle)
Évaluation des modèles SVM, KNN et XGBoost par validation croisée stratifiée sur les groupes A et C.

In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import make_scorer, confusion_matrix

# 1. Charger les labels des participants
participants_df = pd.read_csv('ds004504-download/participants.tsv', sep='\t')
label_map = dict(zip(participants_df['participant_id'], participants_df['Group']))

if 'full_dataset_features' in locals():
    df = full_dataset_features.copy()
    df['target'] = df['subject_id'].map(label_map)

    # FILTRAGE : On garde uniquement A (Alzheimer) et C (Control) pour une classification binaire
    df = df[df['target'].isin(['A', 'C'])]
    print(f"Dataset filtré : {df['target'].value_counts().to_dict()}")

    # Encodage des labels (A -> 1, C -> 0)
    le = LabelEncoder()
    df['target'] = le.fit_transform(df['target'])

    # Features (X) et Target (y)
    X = df.drop(columns=['subject_id', 'epoch_idx', 'target'])
    y = df['target']

    # Gestion des NaNs
    X = X.fillna(X.mean())

    # Normalisation
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Fonctions de score pour classification binaire
    def specificity_score(y_true, y_pred):
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        return tn / (tn + fp) if (tn + fp) > 0 else 0

    scoring = {
        'accuracy': 'accuracy',
        'precision': 'precision',
        'sensitivity': 'recall',
        'f1': 'f1',
        'roc_auc': 'roc_auc',
        'specificity': make_scorer(specificity_score)
    }

    # 3. Initialisation des modèles
    models = {
        'SVM': SVC(probability=True, kernel='rbf'),
        'KNN': KNeighborsClassifier(n_neighbors=5),
        'XGBoost': XGBClassifier(eval_metric='logloss')
    }

    results_list = []
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    print("Début de l'évaluation binaire (A vs C)...")
    for name, model in models.items():
        print(f"Calcul pour {name}...")
        cv_results = cross_validate(model, X_scaled, y, cv=skf, scoring=scoring)

        results_list.append({
            'Model': name,
            'Accuracy': cv_results['test_accuracy'].mean(),
            'Precision': cv_results['test_precision'].mean(),
            'Sensitivity': cv_results['test_sensitivity'].mean(),
            'Specificity': cv_results['test_specificity'].mean(),
            'F1 Score': cv_results['test_f1'].mean(),
            'ROC AUC': cv_results['test_roc_auc'].mean()
        })

    perf_df = pd.DataFrame(results_list)
    display(perf_df)
else:
    print("La matrice 'full_dataset_features' est introuvable.")

Dataset filtré : {'A': 7267, 'C': 6015}
Début de l'évaluation binaire (A vs C)...
Calcul pour SVM...
Calcul pour KNN...
Calcul pour XGBoost...


,Model,Accuracy,Precision,Sensitivity,Specificity,F1 Score,ROC AUC
0,SVM,0.890153,0.909955,0.840732,0.931058,0.873906,0.956047
1,KNN,0.873587,0.862321,0.857855,0.886609,0.860070,0.938439
2,XGBoost,0.913041,0.915805,0.889942,0.932160,0.902637,0.971682


## 6. Classification Multiclasse (A, C, F)
Évaluation de la capacité des modèles à distinguer les trois groupes (Alzheimer, Contrôle, Fronto-temporal).

In [21]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import make_scorer, confusion_matrix

# 1. Préparation des labels
participants_df = pd.read_csv('ds004504-download/participants.tsv', sep='\t')
label_map = dict(zip(participants_df['participant_id'], participants_df['Group']))

if 'full_dataset_features' in locals():
    df_3class = full_dataset_features.copy()
    df_3class['target'] = df_3class['subject_id'].map(label_map)

    print(f"Distribution des 3 classes : {df_3class['target'].value_counts().to_dict()}")

    # Encodage (A, C, F)
    le_3 = LabelEncoder()
    y_3 = le_3.fit_transform(df_3class['target'])
    X_3 = df_3class.drop(columns=['subject_id', 'epoch_idx', 'target']).fillna(df_3class.mean(numeric_only=True))

    # Normalisation
    scaler_3 = StandardScaler()
    X_3_scaled = scaler_3.fit_transform(X_3)

    # Métriques Multiclasse (Macro-average)
    scoring_multiclass = {
        'accuracy': 'accuracy',
        'precision_macro': 'precision_macro',
        'recall_macro': 'recall_macro',
        'f1_macro': 'f1_macro',
        'roc_auc_ovr': 'roc_auc_ovr_weighted'
    }

    models_3 = {
        'SVM (3-class)': SVC(probability=True, kernel='rbf'),
        'KNN (3-class)': KNeighborsClassifier(n_neighbors=5),
        'XGBoost (3-class)': XGBClassifier(eval_metric='mlogloss', num_class=3)
    }

    results_3class = []
    skf_3 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    print("Début de l'évaluation multiclasse (A, C, F)...")
    for name, model in models_3.items():
        print(f"Calcul pour {name}...")
        cv_res = cross_validate(model, X_3_scaled, y_3, cv=skf_3, scoring=scoring_multiclass)

        results_3class.append({
            'Model': name,
            'Accuracy': cv_res['test_accuracy'].mean(),
            'Precision (Macro)': cv_res['test_precision_macro'].mean(),
            'Recall/Sens. (Macro)': cv_res['test_recall_macro'].mean(),
            'F1 Score (Macro)': cv_res['test_f1_macro'].mean(),
            'ROC AUC (OvR)': cv_res['test_roc_auc_ovr'].mean()
        })

    perf_3class_df = pd.DataFrame(results_3class)
    print("\nRésultats de la classification à 3 classes :")
    display(perf_3class_df)
else:
    print("La matrice 'full_dataset_features' est manquante.")

Distribution des 3 classes : {'A': 7267, 'C': 6015, 'F': 4137}
Début de l'évaluation multiclasse (A, C, F)...
Calcul pour SVM (3-class)...
Calcul pour KNN (3-class)...
Calcul pour XGBoost (3-class)...

Résultats de la classification à 3 classes :


,Model,Accuracy,Precision (Macro),Recall/Sens. (Macro),F1 Score (Macro),ROC AUC (OvR)
0,SVM (3-class),0.806361,0.823206,0.782763,0.795494,0.937546
1,KNN (3-class),0.800103,0.800317,0.789682,0.794234,0.921231
2,XGBoost (3-class),0.876974,0.879725,0.870067,0.874337,0.969785
